# Загрузка библиотек

In [ ]:
import pandas as pd
import numpy as np
import chardet
from sqlalchemy import create_engine
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.decomposition import PCA
import pickle
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

# Загрузка датасета

In [32]:
def get_conn(dbname, user, password, host):
    url = f"postgresql+psycopg2://{user}:{password}@{host}/{dbname}"
    return create_engine(url)

In [33]:
dbname = "Regression_data" # Название базы данных
user = "postgres" # Имя пользователя для подключения
password = "86754231qaZ" # Пароль пользователя для подключения
host = "localhost" # Хост


conn = get_conn(dbname, user, password, host)

In [34]:
data = pd.read_sql("SELECT * FROM water_pollution", conn)

# Улучшение модели

## Подготовка данных для модели

In [37]:
# Отбор числовых признаков
cat = ["object"]

cat_data = data.select_dtypes(include=cat)

In [8]:
data

,Year,Contaminant Level (ppm),pH Level,Turbidity (NTU),Dissolved Oxygen (mg/L),Nitrate Level (mg/L),Lead Concentration (µg/L),Bacteria Count (CFU/mL),Access to Clean Water (percent of Population),"Diarrheal Cases per 100,000 people",...,Region_South,Region_West,Water Source Type_Pond,Water Source Type_River,Water Source Type_Spring,Water Source Type_Tap,Water Source Type_Well,Water Treatment Method_Chlorination,Water Treatment Method_Filtration,Water Treatment Method_Nothing
0,2015,6.06,7.12,3.93,4.28,8.28,7.89,3344,33.60,472,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,2017,5.24,7.84,4.79,3.86,15.74,14.68,2122,89.54,122,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,2022,0.24,6.43,0.79,3.42,36.67,9.96,2330,35.29,274,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2016,7.91,6.71,1.96,3.12,36.92,6.77,3779,57.53,3,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,2005,0.12,8.16,4.22,9.15,49.35,12.51,4182,36.60,466,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,2002,2.82,7.40,4.43,9.69,37.58,18.52,359,36.34,12,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2996,2019,8.13,8.33,4.77,7.62,38.05,16.98,3810,81.72,49,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2997,2009,1.18,6.76,4.75,7.07,36.13,7.99,1440,80.11,247,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2998,2009,7.56,6.12,3.49,8.93,25.30,19.86,2919,78.26,232,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [38]:
# Преобразование категориальных признаков в One-Hot Encoded

cat_cols = cat_data.columns # Категориальные признаки

encoder = pickle.load(open("models/Encoder.sav", "rb"))

# Применяем к категориальным колонкам
encoded_data = encoder.transform(data[cat_cols])

# Получаем имена новых колонок
feature_names = encoder.get_feature_names_out(cat_cols)

# Создаем DataFrame с закодированными данными
data_encoded = pd.DataFrame(encoded_data, columns=feature_names)

# Объединяем с числовыми колонками
data = pd.concat([data.drop(cat_cols, axis=1), data_encoded], axis=1)

In [11]:
data

,Year,Contaminant Level (ppm),pH Level,Turbidity (NTU),Dissolved Oxygen (mg/L),Nitrate Level (mg/L),Lead Concentration (µg/L),Bacteria Count (CFU/mL),Access to Clean Water (percent of Population),"Diarrheal Cases per 100,000 people",...,Region_South,Region_West,Water Source Type_Pond,Water Source Type_River,Water Source Type_Spring,Water Source Type_Tap,Water Source Type_Well,Water Treatment Method_Chlorination,Water Treatment Method_Filtration,Water Treatment Method_Nothing
0,2015,6.06,7.12,3.93,4.28,8.28,7.89,3344,33.60,472,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,2017,5.24,7.84,4.79,3.86,15.74,14.68,2122,89.54,122,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,2022,0.24,6.43,0.79,3.42,36.67,9.96,2330,35.29,274,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2016,7.91,6.71,1.96,3.12,36.92,6.77,3779,57.53,3,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,2005,0.12,8.16,4.22,9.15,49.35,12.51,4182,36.60,466,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,2002,2.82,7.40,4.43,9.69,37.58,18.52,359,36.34,12,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2996,2019,8.13,8.33,4.77,7.62,38.05,16.98,3810,81.72,49,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2997,2009,1.18,6.76,4.75,7.07,36.13,7.99,1440,80.11,247,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2998,2009,7.56,6.12,3.49,8.93,25.30,19.86,2919,78.26,232,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [39]:
# Разделение на X и Y
X = data.drop("Population Density (people per km²)", axis=1)
y = data["Population Density (people per km²)"]

In [42]:
X = X.rename(columns={"Access to Clean Water (percent of Population)": "Access to Clean Water (% of Population)"})

In [43]:
# Стандартизация признаков (некоторые модели зависимы от масштаба данных)
scaler = pickle.load(open("models/Scaler.sav", "rb"))
X_scaled = scaler.transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.head()

,Year,Contaminant Level (ppm),pH Level,Turbidity (NTU),Dissolved Oxygen (mg/L),Nitrate Level (mg/L),Lead Concentration (µg/L),Bacteria Count (CFU/mL),Access to Clean Water (% of Population),"Diarrheal Cases per 100,000 people",...,Region_South,Region_West,Water Source Type_Pond,Water Source Type_River,Water Source Type_Spring,Water Source Type_Tap,Water Source Type_Well,Water Treatment Method_Chlorination,Water Treatment Method_Filtration,Water Treatment Method_Nothing
0,0.413295,0.386632,-0.188586,1.021292,-1.091349,-1.158418,-0.372229,0.597773,-1.527319,1.542280,...,-0.489560,-0.496873,-0.420084,-0.467463,-0.464283,-0.44775,-0.44614,-0.576324,1.766601,-0.575811
1,0.689994,0.099878,0.810937,1.627034,-1.298488,-0.644033,0.799011,-0.256066,1.227657,-0.886799,...,-0.489560,2.012587,-0.420084,-0.467463,-0.464283,-0.44775,2.24145,-0.576324,-0.566059,-0.575811
2,1.381740,-1.648621,-1.146462,-1.190369,-1.515490,0.799141,-0.015165,-0.110732,-1.444089,0.168115,...,-0.489560,-0.496873,2.380476,-0.467463,-0.464283,-0.44775,-0.44614,-0.576324,-0.566059,1.736682
3,0.551645,1.033576,-0.757759,-0.366279,-1.663446,0.816380,-0.565423,0.901717,-0.348796,-1.712686,...,-0.489560,-0.496873,-0.420084,-0.467463,-0.464283,-0.44775,2.24145,-0.576324,-0.566059,-0.575811
4,-0.970197,-1.690585,1.255170,1.225554,1.310472,1.673458,0.424697,1.183303,-1.379573,1.500639,...,2.042649,-0.496873,-0.420084,-0.467463,-0.464283,-0.44775,2.24145,-0.576324,1.766601,-0.575811


In [ ]:
# Используем PCA, так как много признаков и мало данных
pca = PCA(n_components=30)

X_pca = pca.fit_transform(X_scaled)

print(sum(pca.explained_variance_ratio_))

0.8452420843689086


In [45]:
pickle.dump(pca, open("models/PCA.sav", "wb"))

Сжимаем кол-во признаков, чтобы избавиться от мультиколлениарности, упростить модель, уменьшить размерность данных. Для этого оставляем 30 компонент, так как сохраняется 84% вариативности.

In [19]:
# Разделение на train и test
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42)

## Градиентный бустинг

In [23]:
gb = GradientBoostingRegressor()
gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)


## Сравнение моделей

In [24]:
#функция для оценки модели
def evaluate_model(y_true, y_pred_list, model_list):
    df_result = pd.DataFrame(columns= ['model','mae','rmse','r2'])
    for i,k in zip(y_pred_list, model_list):
        mae = mean_absolute_error(y_true, i)
        mse = mean_squared_error(y_true, i)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true, i)
        df_result.loc[len(df_result)]=[k,mae,rmse,r2]
    return df_result

In [25]:
df_result = evaluate_model(y_test,[y_pred_gb],['Градиентный бустинг'])
df_result

,model,mae,rmse,r2
0,Градиентный бустинг,251.477135,292.752288,-0.06072


Лучший результат показала модель GradientBoostingRegressor

## Оптимизация параметров модели

In [26]:
gb = GradientBoostingRegressor(random_state=42)

param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'min_samples_split': [2, 5]
}

In [27]:
scoring = {
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MSE': make_scorer(mean_squared_error, greater_is_better=False),
    'R2': make_scorer(r2_score)
}

In [28]:
grid_search = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    scoring=scoring,
    refit='R2',
    cv=3, 
    n_jobs=-1,  
    verbose=2  
)

grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 54 candidates, totalling 162 fits
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=100; total time=   3.3s
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=100; total time=   2.8s
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=100; total time=   2.9s
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=200; total time=   5.5s
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=200; total time=   6.1s
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=200; total time=   6.5s
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=300; total time=   8.7s
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=300; total time=  15.0s
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=300; total time=   9.1s
[CV] END learning_rate=0.01, max_depth

GridSearchCV(cv=3, estimator=GradientBoostingRegressor(random_state=42),
             n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.05, 0.1],
                         'max_depth': [3, 4, 5], 'min_samples_split': [2, 5],
                         'n_estimators': [100, 200, 300]},
             refit='R2',
             scoring={'MAE': make_scorer(mean_absolute_error, greater_is_better=False, response_method='predict'),
                      'MSE': make_scorer(mean_squared_error, greater_is_better=False, response_method='predict'),
                      'R2': make_scorer(r2_score, response_method='predict')},
             verbose=2)

In [29]:
print("Лучшие параметры:", grid_search.best_params_)
print("Лучший R2-score:", grid_search.best_score_)

Лучшие параметры: {'learning_rate': 0.01, 'max_depth': 3, 'min_samples_split': 2, 'n_estimators': 100}
Лучший R2-score: -0.012363965389124912


In [30]:
pickle.dump(grid_search, open("models/GradientBoostingRegressor.sav", 'wb'))